# SAC arrival_v2 — s0 sensor envelope on production benchmarks (seed=42, 1M, vanilla)

**前情**：commit `f00074e` / `f771414` 已闭环 arrival_v2 reward 在 **s1 / 4 个 production benchmark** 上的 vanilla SAC 严格控制对照（详见 [`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7）：

| §7 cell | benchmark | sensor | seed | total_steps | gate |
|---|---|---|---|---:|---|
| §7.1 | `single_u15_cross_tgt15` | **s1** / 12-D | 42 | 1M | PASS（borderline, 3/30 OOB） |
| §7.2 | `single_u15_upstream_tgt15` | **s1** / 12-D | 42 | 1M | PASS |
| §7.3 | `tandem_u15_upstream_tgt15` | **s1** / 12-D | 42 | 1M | PASS |
| §7.4 | `sbs_u15_upstream_tgt15` | **s1** / 12-D | 42 | 1M | PASS |

**本 notebook 任务（严格 sensor 对照）**：把 §7 的 4 cell **唯一变量**从 sensor 切到 `s0`（DVL water-track only, deployment-realistic, 10-D 观测），其它**全部不动**：

| 维度 | §7（已落） | 本 notebook |
|---|---|---|
| sensor layout | `s1_k4` (12-D) | **`s0_k4` (10-D)** ← 唯一变量 |
| reward | arrival_v2 | arrival_v2 |
| flow U / target | 1.5 / 1.5 | 1.5 / 1.5 |
| seed | 42 | 42 |
| total_steps | 1M | 1M |
| num_envs | 6 | 6 |
| algorithm | vanilla SAC | vanilla SAC |
| 4 个 benchmark | cross / upstream / tandem / sbs | 同 4 个 |

跑完后，§7 的表格在 sensor 轴上从单列扩展到 s0 × s1 两列，**任何差异只能归因于 sensor 信息差异**。

**为什么不引入 improved SAC？** 见 chat 决策记录：A0 evidence（[`docs/online_rl_line_summary.md`](../docs/online_rl_line_summary.md) §1.1）显示 s0 + arrival_v1 + cross_u10 已达 0.967（near-saturation），且 s0-s1 gap 是 reward-dependent（efficiency_v2 = 18pp，arrival_v1 = 3pp）。本次先把 vanilla 在 s0 上的 production-difficulty envelope 跑出来，再决定后续是否值得跑 improved SAC。

**4 phase 顺序（ascending difficulty per s1 evidence）**：

| Phase | benchmark | s1 baseline | 预期 |
|---|---|---:|---|
| 1 | `tandem_u15_upstream_tgt15` | 1.000 | 大概率仍 PASS |
| 2 | `sbs_u15_upstream_tgt15` | 1.000 | 大概率仍 PASS |
| 3 | `single_u15_upstream_tgt15` | 1.000 | 可能下降 |
| 4 | `single_u15_cross_tgt15` | 0.900（borderline） | **最可能下降**；s0 在 cross 几何下信息流入最不利 |

**Gate**（每个 phase 独立判定，与 §7 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Single seed (=42)**：与 §7 同步。本 notebook 输出**是 exploratory**，不是 thesis-grade。如果 4 phase 全 PASS 且各指标接近 s1 → 论点是 "arrival_v2 在 s0 deployable sensor + production difficulty 下也 work"；如果 phase 4 (single_cross) fail 或者某 phase 显著低于 s1 → 论点是 "sensor 信息差异在某些 geometry / topology 下确实非零"，下一步可考虑 multi-seed 复现或 improved SAC ablation。

**输出根（与 §7 平行；s1_k4 → s0_k4，互不覆盖）**：
- Phase 1: `experiments/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42/`
- Phase 2: `experiments/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42/`
- Phase 3: `experiments/arrival_v2_prototype/single_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42/`
- Phase 4: `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42/`

**总预算**：4 × ~2.5h L4 ≈ 10h（1-2 Colab Pro+ session）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout），与 §7 系列 notebook 一致。


## 0. GPU sanity


In [ ]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')


## 1. Mount Drive + cwd

与 §7 系列（`f00074e` / `f771414`）同一个 working clone。


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


## 2. 通用配置 — 4 个 phase 共用

唯一变量是 `benchmark_key` + `task_geometry` + `flow_path`。所有 SAC / env / reward 超参与 §7 完全一致，**除了 `PROBE_LAYOUT = 's0'`**——这是本 notebook 的唯一对照变量。

若某 phase 1.0M 不够，把对应的 `*_TOTAL_STEPS` 改大后重跑该 phase 的 config + train cell；skip/resume 自动续训。


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== 全局 SAC / env config (4 个 phase 共用) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'                  # ← 唯一与 §7 不同的配置
HISTORY_LENGTH = 4
TARGET_SPEED = 1.5
SEED = 42

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow files (与 §7 严格一致)
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
TANDEM_FLOW = 'wake_data/wake_tandem_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'
SBS_FLOW    = 'wake_data/wake_sbs_G35_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Phase 1 — tandem (T_*) ====
T_BENCHMARK_KEY = 'tandem_u15_upstream_tgt15'
T_TASK_GEOMETRY = 'upstream'
T_FLOW_PATH = TANDEM_FLOW
T_TOTAL_STEPS = 1_000_000
T_RUN_ROOT = Path('experiments/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
T_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/tandem_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
T_MANIFEST_PATH = Path(f'benchmarks/{T_BENCHMARK_KEY}.json')

# ==== Phase 2 — side-by-side (S_*) ====
S_BENCHMARK_KEY = 'sbs_u15_upstream_tgt15'
S_TASK_GEOMETRY = 'upstream'
S_FLOW_PATH = SBS_FLOW
S_TOTAL_STEPS = 1_000_000
S_RUN_ROOT = Path('experiments/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
S_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/sbs_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
S_MANIFEST_PATH = Path(f'benchmarks/{S_BENCHMARK_KEY}.json')

# ==== Phase 3 — single_upstream (U_*) ====
U_BENCHMARK_KEY = 'single_u15_upstream_tgt15'
U_TASK_GEOMETRY = 'upstream'
U_FLOW_PATH = SINGLE_FLOW
U_TOTAL_STEPS = 1_000_000
U_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
U_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_upstream_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
U_MANIFEST_PATH = Path(f'benchmarks/{U_BENCHMARK_KEY}.json')

# ==== Phase 4 — single_cross (C_*) ====
C_BENCHMARK_KEY = 'single_u15_cross_tgt15'
C_TASK_GEOMETRY = 'cross_stream'
C_FLOW_PATH = SINGLE_FLOW
C_TOTAL_STEPS = 1_000_000
C_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
C_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')
C_MANIFEST_PATH = Path(f'benchmarks/{C_BENCHMARK_KEY}.json')

# Cast Paths to strings for shell command interpolation
PHASE_CONFIGS = [
    ('TANDEM   (Phase 1)', T_BENCHMARK_KEY, T_FLOW_PATH, T_TASK_GEOMETRY, T_TOTAL_STEPS, T_RUN_ROOT, T_CKPT_ROOT, T_MANIFEST_PATH),
    ('SBS      (Phase 2)', S_BENCHMARK_KEY, S_FLOW_PATH, S_TASK_GEOMETRY, S_TOTAL_STEPS, S_RUN_ROOT, S_CKPT_ROOT, S_MANIFEST_PATH),
    ('UPSTREAM (Phase 3)', U_BENCHMARK_KEY, U_FLOW_PATH, U_TASK_GEOMETRY, U_TOTAL_STEPS, U_RUN_ROOT, U_CKPT_ROOT, U_MANIFEST_PATH),
    ('CROSS    (Phase 4)', C_BENCHMARK_KEY, C_FLOW_PATH, C_TASK_GEOMETRY, C_TOTAL_STEPS, C_RUN_ROOT, C_CKPT_ROOT, C_MANIFEST_PATH),
]

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT  = {PROBE_LAYOUT}  (← 唯一与 §7 不同的配置)')
print(f'OBJECTIVE     = {OBJECTIVE}')
print(f'HISTORY       = k={HISTORY_LENGTH}')
print(f'SEED          = {SEED}')
print(f'NUM_ENVS      = {NUM_ENVS}')
print(f'EVAL_EVERY    = {EVAL_EVERY:,}  EVAL_EPISODES = {EVAL_EPISODES}')
print()
print(f'{"Phase":<22}{"benchmark":<32}{"geometry":<14}{"total_steps":>12}')
print('-' * 80)
for label, bk, _fp, geom, ts, _rr, _ck, _mp in PHASE_CONFIGS:
    print(f'{label:<22}{bk:<32}{geom:<14}{ts:>12,}')
print()
print(f'all run_root under: experiments/arrival_v2_prototype/<benchmark>/arrival_v2/sac_vanilla/s0_k4/seed_42/')
print(f'parallel to §7 tree which is at s1_k4/seed_42/ — 互不覆盖')


## 3. Preflight — flow files / arrival_v2 candidate gate / reward unit tests / manifests

停止条件：
- 任一 flow 缺失 → raise
- `validate_arrival_v2_candidate` 失败 → reward 公式不满足 invariants
- `test_reward_objective.py` 失败 → reward 实现退化
- 任一 manifest 生成失败 → eval 不可重复


In [ ]:
# 检查 3 个独立 flow 文件
for label, path in [('single', SINGLE_FLOW), ('tandem', TANDEM_FLOW), ('sbs', SBS_FLOW)]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f'missing {label} flow file: {p}')
    print(f'[OK] {label:<8} flow: {p}  ({p.stat().st_size / 1e6:.1f} MB)')


In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate


In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q


In [ ]:
for key, mp in [
    (T_BENCHMARK_KEY, T_MANIFEST_PATH),
    (S_BENCHMARK_KEY, S_MANIFEST_PATH),
    (U_BENCHMARK_KEY, U_MANIFEST_PATH),
    (C_BENCHMARK_KEY, C_MANIFEST_PATH),
]:
    if not mp.exists():
        !python -u -m scripts.generate_standard_benchmarks --benchmarks {key} --episodes {EVAL_EPISODES}
    if not mp.exists():
        raise FileNotFoundError(f'manifest not generated: {mp}')
    print(f'[OK] manifest ready: {mp}')


## 4. Phase 1 — tandem train (1.0M, skip/resume)

Skip/resume 逻辑：
- `env_step >= T_TOTAL_STEPS` → skip
- `0 < env_step < T_TOTAL_STEPS` → `--resume` 续训
- `env_step == 0` → fresh start

要延长训练：把 `T_TOTAL_STEPS` 改大，重跑 config cell + 本 cell。


In [ ]:
t_state_path = T_RUN_ROOT / 'trainer_state.json'
if t_state_path.exists():
    t_state = json.loads(t_state_path.read_text(encoding='utf-8'))
    t_current_step = int(t_state.get('env_step', 0))
else:
    t_current_step = 0
print(f'[state] T env_step = {t_current_step:,} / target {T_TOTAL_STEPS:,}')

if t_current_step >= T_TOTAL_STEPS:
    print(f'[skip] T already trained to {t_current_step:,} >= {T_TOTAL_STEPS:,}')
elif t_current_step > 0:
    print(f'[resume] T continuing from {t_current_step:,} -> {T_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(T_RUN_ROOT)} \
        --total-steps {T_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(T_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] T fresh start -> {T_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {T_FLOW_PATH} \
        --task-geometry {T_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {T_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(T_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(T_RUN_ROOT)} \
        --checkpoint-dir {str(T_CKPT_ROOT)}


## 5. Phase 1 — tandem summary + gate


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / {PROBE_LAYOUT} / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

t_summary = summarize_phase(
    T_RUN_ROOT,
    T_TOTAL_STEPS,
    'TANDEM_VALIDATION',
    'tandem_validation_gate_summary.json',
)
T_PASS = t_summary['all_pass']
print()
print(f'T_PASS = {T_PASS}')


## 6. Phase 2 — side-by-side train (1.0M, skip/resume)

Skip/resume 逻辑：
- `env_step >= S_TOTAL_STEPS` → skip
- `0 < env_step < S_TOTAL_STEPS` → `--resume` 续训
- `env_step == 0` → fresh start

要延长训练：把 `S_TOTAL_STEPS` 改大，重跑 config cell + 本 cell。


In [ ]:
s_state_path = S_RUN_ROOT / 'trainer_state.json'
if s_state_path.exists():
    s_state = json.loads(s_state_path.read_text(encoding='utf-8'))
    s_current_step = int(s_state.get('env_step', 0))
else:
    s_current_step = 0
print(f'[state] S env_step = {s_current_step:,} / target {S_TOTAL_STEPS:,}')

if s_current_step >= S_TOTAL_STEPS:
    print(f'[skip] S already trained to {s_current_step:,} >= {S_TOTAL_STEPS:,}')
elif s_current_step > 0:
    print(f'[resume] S continuing from {s_current_step:,} -> {S_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(S_RUN_ROOT)} \
        --total-steps {S_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(S_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] S fresh start -> {S_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {S_FLOW_PATH} \
        --task-geometry {S_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {S_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(S_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(S_RUN_ROOT)} \
        --checkpoint-dir {str(S_CKPT_ROOT)}


## 7. Phase 2 — side-by-side summary + gate


In [ ]:
s_summary = summarize_phase(
    S_RUN_ROOT,
    S_TOTAL_STEPS,
    'SBS_VALIDATION',
    'sbs_validation_gate_summary.json',
)
S_PASS = s_summary['all_pass']
print()
print(f'S_PASS = {S_PASS}')


## 8. Phase 3 — single_upstream train (1.0M, skip/resume)

Skip/resume 逻辑：
- `env_step >= U_TOTAL_STEPS` → skip
- `0 < env_step < U_TOTAL_STEPS` → `--resume` 续训
- `env_step == 0` → fresh start

要延长训练：把 `U_TOTAL_STEPS` 改大，重跑 config cell + 本 cell。


In [ ]:
u_state_path = U_RUN_ROOT / 'trainer_state.json'
if u_state_path.exists():
    u_state = json.loads(u_state_path.read_text(encoding='utf-8'))
    u_current_step = int(u_state.get('env_step', 0))
else:
    u_current_step = 0
print(f'[state] U env_step = {u_current_step:,} / target {U_TOTAL_STEPS:,}')

if u_current_step >= U_TOTAL_STEPS:
    print(f'[skip] U already trained to {u_current_step:,} >= {U_TOTAL_STEPS:,}')
elif u_current_step > 0:
    print(f'[resume] U continuing from {u_current_step:,} -> {U_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(U_RUN_ROOT)} \
        --total-steps {U_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(U_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] U fresh start -> {U_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {U_FLOW_PATH} \
        --task-geometry {U_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {U_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(U_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(U_RUN_ROOT)} \
        --checkpoint-dir {str(U_CKPT_ROOT)}


## 9. Phase 3 — single_upstream summary + gate


In [ ]:
u_summary = summarize_phase(
    U_RUN_ROOT,
    U_TOTAL_STEPS,
    'SINGLE_UPSTREAM_VALIDATION',
    'single_upstream_validation_gate_summary.json',
)
U_PASS = u_summary['all_pass']
print()
print(f'U_PASS = {U_PASS}')


## 10. Phase 4 — single_cross train (1.0M, skip/resume)

Skip/resume 逻辑：
- `env_step >= C_TOTAL_STEPS` → skip
- `0 < env_step < C_TOTAL_STEPS` → `--resume` 续训
- `env_step == 0` → fresh start

要延长训练：把 `C_TOTAL_STEPS` 改大，重跑 config cell + 本 cell。


In [ ]:
c_state_path = C_RUN_ROOT / 'trainer_state.json'
if c_state_path.exists():
    c_state = json.loads(c_state_path.read_text(encoding='utf-8'))
    c_current_step = int(c_state.get('env_step', 0))
else:
    c_current_step = 0
print(f'[state] C env_step = {c_current_step:,} / target {C_TOTAL_STEPS:,}')

if c_current_step >= C_TOTAL_STEPS:
    print(f'[skip] C already trained to {c_current_step:,} >= {C_TOTAL_STEPS:,}')
elif c_current_step > 0:
    print(f'[resume] C continuing from {c_current_step:,} -> {C_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(C_RUN_ROOT)} \
        --total-steps {C_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(C_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] C fresh start -> {C_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {C_FLOW_PATH} \
        --task-geometry {C_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {C_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(C_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(C_RUN_ROOT)} \
        --checkpoint-dir {str(C_CKPT_ROOT)}


## 11. Phase 4 — single_cross summary + gate


In [ ]:
c_summary = summarize_phase(
    C_RUN_ROOT,
    C_TOTAL_STEPS,
    'SINGLE_CROSS_VALIDATION',
    'single_cross_validation_gate_summary.json',
)
C_PASS = c_summary['all_pass']
print()
print(f'C_PASS = {C_PASS}')


## 12. Combined verdict + §7 sensor envelope diff

每个 phase 独立判定；汇总 4 phase 的 5/5 gate 状态，并打印 s0 vs §7 (s1) 的逐项差值（人读用）。

**对照预期** — 完成后写入 `arrival_v2_experiment_report.md` §7 新增 sensor envelope 子节，4 × 2 表（4 benchmark × {s0_k4, s1_k4}）一次性更新。任何 phase fail 都是 actionable signal：
- `single_cross_s0` fail → 反映 s0 在 cross 几何下信息流入不足（横流 + 单点 DVL，agent 无法预见涡街）
- `single_upstream_s0` fail 但 cross_s0 PASS → 反常；需要检查 reward shaping 在 upstream 下是否对 s0 不友好
- 4 phase 全部 PASS 且接近 s1 数值 → arrival_v2 在 deployable s0 + production benchmarks 下完整 work；这是 ReBRAC paper offline 主线 deployable-only 协议的 online cross-reference baseline


In [ ]:
print('=' * 96)
print('S0 SENSOR ENVELOPE — combined verdict')
print('-' * 96)

summaries = {
    'TANDEM        ': t_summary,
    'SBS           ': s_summary,
    'SINGLE_UPSTRM ': u_summary,
    'SINGLE_CROSS  ': c_summary,
}

# §7 (s1) baselines for diff column (manually transcribed from f00074e §7.5 table)
s1_baselines = {
    'TANDEM        ': {'final': 1.000, 'oob': 0.000},
    'SBS           ': {'final': 1.000, 'oob': 0.000},
    'SINGLE_UPSTRM ': {'final': 1.000, 'oob': 0.000},
    'SINGLE_CROSS  ': {'final': 0.900, 'oob': 0.100},
}

print(f'{"phase":<16}{"pass":>6}{"final":>10}{"peak":>10}{"@step":>10}{"oob":>8}    Δ vs §7 (s1)')
print('-' * 96)
for label, summ in summaries.items():
    bl = s1_baselines[label]
    dfinal = summ['final_success_rate'] - bl['final']
    doob = summ['final_oob_rate'] - bl['oob']
    print(f'{label}{str(summ["all_pass"]):>6}'
          f'{summ["final_success_rate"]:>10.4f}'
          f'{summ["peak_success_rate"]:>10.4f}'
          f'{summ["peak_step"]:>10,}'
          f'{summ["final_oob_rate"]:>8.4f}'
          f'    Δfinal={dfinal:+.4f}  Δoob={doob:+.4f}')
print('=' * 96)

all_pass = all(s['all_pass'] for s in summaries.values())
print(f'\n4-phase all_pass = {all_pass}')

combined = {
    'experiment': 'arrival_v2_s0_sensor_envelope',
    'commit_baseline': 'f771414',
    'seed': SEED,
    'probe_layout': PROBE_LAYOUT,
    'tandem': t_summary,
    'sbs': s_summary,
    'single_upstream': u_summary,
    'single_cross': c_summary,
    's1_baselines_referenced': s1_baselines,
    'all_pass': bool(all_pass),
}
out_dir = Path('experiments/arrival_v2_prototype/s0_sensor_envelope_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(combined, indent=2), encoding='utf-8')
print(f'[saved] {out_path}')
